# Test conversion PDF → Word

Notebook interactif pour tester les **8 moteurs** de conversion PDF → Word de `docpipeline.conversion`
sur la fixture `tests/fixtures/notice_garanties.pdf`.

Les moteurs absents de la machine (Adobe, MS Word, LibreOffice, Docling, Tesseract) sont sautés
automatiquement. Chaque cellule produit un `.docx` dans `notebooks/_outputs/` que tu peux ouvrir
pour comparer visuellement.

**Ordre conseillé d'exécution :** cellules 1 → fin (chaque section est indépendante après le
*Setup*).

## 1. Setup

In [ ]:
from __future__ import annotations

import os, sys, time, shutil
from pathlib import Path

# Force UTF-8 sur la console Windows (sinon caractères accentués cassés)
if sys.platform == "win32":
    os.environ.setdefault("PYTHONUTF8", "1")
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

# Permettre l'import depuis le repo même sans `pip install -e .`
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PDF_FILE = REPO_ROOT / "tests" / "fixtures" / "notice_garanties.pdf"
OUT_DIR  = REPO_ROOT / "notebooks" / "_outputs"
OUT_DIR.mkdir(exist_ok=True)

assert PDF_FILE.exists(), f"Fixture introuvable : {PDF_FILE}"
print(f"PDF source : {PDF_FILE}")
print(f"Sortie     : {OUT_DIR}")

In [ ]:
from docpipeline.conversion import (
    AdobeConverter,
    DoclingConverter,
    DocxEnhancer,
    HybridConverter,
    LibreOfficeConverter,
    MSWordConverter,
    OCRConverter,
    SmartConverter,
    TextConverter,
    convert_pdf_to_word,
    ConversionResult,
)
from docpipeline.conversion._adobe_credentials import adobe_credentials_available
from docpipeline.conversion._libreoffice_converter import _find_libreoffice
from docpipeline.parsing.pdf import classify_pdf

In [ ]:
# Helper : appel d'un convertisseur isolé avec timing + skip propre
def run_converter(name: str, converter, *, available: bool = True, **kwargs):
    out = OUT_DIR / f"out_{name}.docx"
    if not available:
        print(f"⏭  {name:14s} — indisponible (sauté)")
        return None
    try:
        t0 = time.perf_counter()
        result = converter.convert(PDF_FILE, out, **kwargs)
        dt = time.perf_counter() - t0
        size_kb = Path(result).stat().st_size / 1024
        print(f"✅ {name:14s} — {dt:5.1f}s, {size_kb:7.1f} KB → {Path(result).name}")
        return Path(result)
    except Exception as exc:
        print(f"❌ {name:14s} — {type(exc).__name__}: {exc}")
        return None

# Détection des moteurs optionnels disponibles sur cette machine
ADOBE_OK       = adobe_credentials_available()
LIBREOFFICE_OK = _find_libreoffice() is not None
MSWORD_OK      = sys.platform == "win32"  # tentative ; échec capturé à l'usage
TESSERACT_OK   = shutil.which("tesseract") is not None
try:
    import docling  # noqa: F401
    DOCLING_OK = True
except ImportError:
    DOCLING_OK = False

print("Moteurs disponibles :")
print(f"  adobe       : {ADOBE_OK}")
print(f"  msword      : {MSWORD_OK}  (Windows uniquement, MS Office requis)")
print(f"  libreoffice : {LIBREOFFICE_OK}")
print(f"  docling     : {DOCLING_OK}")
print(f"  ocr         : {TESSERACT_OK}")

## 2. Classification du PDF source

La classification détermine le moteur choisi en mode auto. Catégories possibles :
`word_native` / `design_tool` / `scanned` / `other`.

In [ ]:
info = classify_pdf(PDF_FILE)
print(f"Catégorie  : {info.category.value}")
print(f"Confiance  : {info.confidence:.2f}")
print(f"Créateur   : {info.creator}")
print(f"Producteur : {info.producer}")
print(f"Pages      : {info.page_count}")
print(f"Signaux    : {info.signals}")

## 3. Moteurs gratuits, toujours disponibles

In [ ]:
# pdf2docx — optimal pour PDFs Word natifs
_ = run_converter("text", TextConverter())

In [ ]:
# PyMuPDF reconstruction — fallback offline
_ = run_converter("smart", SmartConverter())

In [ ]:
# Image + texte invisible — visuel parfait, NON éditable
_ = run_converter("hybrid", HybridConverter(), dpi=200)

## 4. Moteurs optionnels (avec skip si non installés)

In [ ]:
# Adobe PDF Services API — qualité Acrobat Pro (cloud, 500 conv./mois gratuites)
_ = run_converter("adobe", AdobeConverter(), available=ADOBE_OK)

In [ ]:
# MS Word PDF Reflow — Windows + Office
_ = run_converter("msword", MSWordConverter(), available=MSWORD_OK)

In [ ]:
# LibreOffice headless — multi-OS
_ = run_converter("libreoffice", LibreOfficeConverter(), available=LIBREOFFICE_OK)

In [ ]:
# Docling — IBM ML, gratuit + offline (~80% qualité Adobe)
_ = run_converter("docling", DoclingConverter(), available=DOCLING_OK)

In [ ]:
# OCR Tesseract — PDFs scannés
_ = run_converter("ocr", OCRConverter(), available=TESSERACT_OK, lang="fra+eng")

## 5. `convert_pdf_to_word` — sélection automatique

C'est le point d'entrée haut niveau : il classe le PDF, choisit le meilleur moteur disponible,
applique l'enhancement, et renvoie un `ConversionResult` riche.

In [ ]:
def show_result(label: str, r: ConversionResult) -> None:
    print(f"\n─ {label}")
    print(f"  output         : {r.output_path.name}")
    print(f"  engine_used    : {r.engine_used}")
    print(f"  category       : {r.category.value}  (confiance {r.confidence:.2f})")
    print(f"  enhanced       : {r.enhanced}")
    print(f"  editable       : {r.editable}")
    print(f"  visual_fidelity: {r.visual_fidelity}")
    if r.warnings:
        print("  warnings :")
        for w in r.warnings:
            print(f"    • {w}")

result_auto = convert_pdf_to_word(PDF_FILE, OUT_DIR / "out_auto.docx")
show_result("AUTO (cascade complète)", result_auto)

## 6. `force_engine` — forcer un moteur spécifique

In [ ]:
engines_to_test = [
    ("smart",       True),
    ("text",        True),
    ("hybrid",      True),
    ("adobe",       ADOBE_OK),
    ("msword",      MSWORD_OK),
    ("libreoffice", LIBREOFFICE_OK),
    ("docling",     DOCLING_OK),
    ("ocr",         TESSERACT_OK),
]

for engine, available in engines_to_test:
    if not available:
        print(f"⏭  force_engine={engine!r:14s} — indisponible (sauté)")
        continue
    try:
        out = OUT_DIR / f"out_force_{engine}.docx"
        r = convert_pdf_to_word(
            PDF_FILE, out, force_engine=engine, enhance=False,
        )
        size_kb = r.output_path.stat().st_size / 1024
        print(f"✅ force_engine={engine!r:14s} → {r.engine_used:30s} "
              f"editable={r.editable}, fidelity={r.visual_fidelity}, {size_kb:.1f} KB")
    except Exception as exc:
        print(f"❌ force_engine={engine!r:14s} — {type(exc).__name__}: {exc}")

## 7. Stratégies `prefer` (balanced / editable / visual)

- `balanced` (défaut) : meilleur compromis fidélité ↔ éditabilité.
- `editable`          : ne choisit **jamais** `hybrid`.
- `visual`            : peut tomber sur `hybrid` si aucun moteur premium n'est dispo.

In [ ]:
for prefer in ("balanced", "editable", "visual"):
    out = OUT_DIR / f"out_prefer_{prefer}.docx"
    r = convert_pdf_to_word(PDF_FILE, out, prefer=prefer)
    show_result(f"prefer={prefer!r}", r)

## 8. Flag `enhance` — post-traitement DocxEnhancer (11 étapes)

L'enhancement n'est appliqué qu'aux moteurs qui en ont besoin (smart, text, ocr).
Adobe / MSWord / LibreOffice / Docling / Hybrid produisent déjà du DOCX propre.

In [ ]:
out_raw = OUT_DIR / "out_smart_raw.docx"
out_enh = OUT_DIR / "out_smart_enhanced.docx"

r_raw = convert_pdf_to_word(PDF_FILE, out_raw, force_engine="smart", enhance=False)
r_enh = convert_pdf_to_word(PDF_FILE, out_enh, force_engine="smart", enhance=True)

print(f"Sans enhance : {out_raw.stat().st_size / 1024:.1f} KB  enhanced={r_raw.enhanced}")
print(f"Avec enhance : {out_enh.stat().st_size / 1024:.1f} KB  enhanced={r_enh.enhanced}")

## 9. `DocxEnhancer` en standalone

Permet d'améliorer un DOCX existant sans repasser par la conversion.

In [ ]:
src_docx = OUT_DIR / "enh_src.docx"
SmartConverter().convert(PDF_FILE, src_docx)

out = OUT_DIR / "enh_out.docx"
DocxEnhancer().enhance(src_docx, output_path=out, source_pdf_path=PDF_FILE)

print(f"Source   : {src_docx.stat().st_size / 1024:.1f} KB")
print(f"Enhanced : {out.stat().st_size / 1024:.1f} KB")

## 10. Récapitulatif des fichiers générés

In [ ]:
files = sorted(OUT_DIR.glob("*.docx"), key=lambda p: p.name)
print(f"{len(files)} fichiers dans {OUT_DIR.relative_to(REPO_ROOT)}/ :\n")
for f in files:
    print(f"  {f.stat().st_size / 1024:8.1f} KB  {f.name}")

### Nettoyage (optionnel)

Dé-commenter pour supprimer les fichiers générés.

In [ ]:
# import shutil
# shutil.rmtree(OUT_DIR, ignore_errors=True)
# print("Sortie nettoyée")